# NB04c — Post-Training Evaluation (YOLO11n)

> Run this notebook **after NB04 training completes**.  
> It auto-detects the latest `ppe_yolo11n_1280*` run folder, so no paths to edit.

**What this notebook produces:**
1. Training curve plots (6-panel: losses + mAP50 + recall + precision)
2. YOLO’s built-in plots (F1, PR, confusion matrix, sample predictions)
3. Per-class AP50 checkpoint eval at key epochs (ep5, ep20, best)
4. Minority-class ablation decision gate
5. Final evaluation on **val + test** splits → saved to `nb07_model_comparison.csv`

In [ ]:
# ============================================================
# 0. SETUP — run this cell first
# ============================================================
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image as IPImage

from ultralytics import YOLO

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# ---- Paths ----
PROJECT_ROOT = Path('../')
DATA_YAML    = PROJECT_ROOT / 'data' / 'ppe_dataset.yaml'
CFG_JSON     = PROJECT_ROOT / 'data' / 'training_config.json'
RESULTS_DIR  = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# ---- Auto-detect RUNS_DIR (YOLO may save under notebooks/ or project root) ----
_candidate_dirs = [
    Path('runs/detect'),                  # CWD = notebooks/
    PROJECT_ROOT / 'runs' / 'detect',     # project root
]
RUNS_DIR = None
for _d in _candidate_dirs:
    if _d.is_dir() and any(_d.iterdir()):
        RUNS_DIR = _d
        break

assert RUNS_DIR is not None, (
    'Could not find runs/detect with training results. '
    'Checked: ' + ', '.join(str(d.resolve()) for d in _candidate_dirs)
)

CLASS_NAMES  = ['hardhat', 'no-hardhat', 'vest', 'no-vest', 'person']
MINORITY_CLS = ['vest', 'no-vest']

with open(CFG_JSON) as f:
    cfg = json.load(f)['yolo11n']

# ---- Auto-detect the latest completed run ----
candidates = sorted(
    [p for p in RUNS_DIR.iterdir() if p.name.startswith('ppe_yolo11n_1280')],
    key=lambda p: p.stat().st_mtime, reverse=True
)
assert len(candidates) > 0, f'No ppe_yolo11n_1280* runs found in {RUNS_DIR.resolve()}'
RUN_DIR     = candidates[0]
RUN_NAME    = RUN_DIR.name
WEIGHTS_DIR = RUN_DIR / 'weights'

print(f'Runs dir    : {RUNS_DIR.resolve()}')
print(f'Run folder  : {RUN_DIR.resolve()}')
print(f'Data YAML   : {DATA_YAML.resolve()}')
print(f'Results dir : {RESULTS_DIR.resolve()}')
print()
print('Weights available:')
for w in sorted(WEIGHTS_DIR.glob('*.pt')):
    print(f'  {w.name}')

---
## 1. Training Curves (from results.csv)

In [ ]:
# ============================================================
# 1. TRAINING CURVES — 6-panel plot
# ============================================================
results_csv = RUN_DIR / 'results.csv'
assert results_csv.exists(), f'results.csv not found at {results_csv}'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

best_ep = int(df['metrics/mAP50(B)'].idxmax()) + 1
best_map50 = df['metrics/mAP50(B)'].max()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f'Training Curves \u2014 {RUN_NAME}\n'
    f'{len(df)} epochs  |  Best mAP50 = {best_map50:.3f} @ epoch {best_ep}',
    fontsize=14, fontweight='bold'
)

plot_configs = [
    ('train/cls_loss',       axes[0,0], 'Classification Loss',  'cls_loss',  1.5),
    ('train/box_loss',       axes[0,1], 'Box Regression Loss',  'box_loss',  None),
    ('train/dfl_loss',       axes[0,2], 'DFL Loss',             'dfl_loss',  None),
    ('metrics/mAP50(B)',     axes[1,0], 'mAP50 (val)',          'mAP50',     0.55),
    ('metrics/recall(B)',    axes[1,1], 'Recall (val)',          'Recall',    0.50),
    ('metrics/precision(B)', axes[1,2], 'Precision (val)',       'Precision', None),
]

epochs = df.index + 1
for col, ax, title, ylabel, alert in plot_configs:
    if col not in df.columns:
        ax.set_title(f'{title}\n(not in results.csv)')
        continue
    ax.plot(epochs, df[col], linewidth=2, color='steelblue')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    if alert is not None:
        ax.axhline(alert, color='red', linestyle='--', alpha=0.7,
                   label=f'Alert ({alert})')
        ax.legend(fontsize=9)
    for ckpt_ep in [5, 10, 20, 50]:
        if ckpt_ep <= len(epochs):
            ax.axvline(ckpt_ep, color='orange', linestyle=':', alpha=0.5)

plt.tight_layout()
plot_path = RESULTS_DIR / f'nb04_training_curves_ep{len(df)}.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

# ---- Summary numbers ----
latest = df.iloc[-1]
print(f'\nFinal epoch ({len(df)}):')
print(f'  mAP50     : {latest.get("metrics/mAP50(B)", 0):.4f}')
print(f'  mAP50-95  : {latest.get("metrics/mAP50-95(B)", 0):.4f}')
print(f'  Precision : {latest.get("metrics/precision(B)", 0):.4f}')
print(f'  Recall    : {latest.get("metrics/recall(B)", 0):.4f}')
print(f'  box_loss  : {latest.get("train/box_loss", 0):.4f}')
print(f'  cls_loss  : {latest.get("train/cls_loss", 0):.4f}')
print(f'  dfl_loss  : {latest.get("train/dfl_loss", 0):.4f}')

---
## 2. YOLO Built-in Plots (F1, PR, Confusion Matrix, Predictions)

In [ ]:
# ============================================================
# 2. YOLO BUILT-IN PLOTS
# ============================================================
plot_files = [
    ('results.png',                    'Ultralytics Training Summary'),
    ('F1_curve.png',                   'F1-Confidence Curve'),
    ('PR_curve.png',                   'Precision-Recall Curve'),
    ('P_curve.png',                    'Precision-Confidence Curve'),
    ('R_curve.png',                    'Recall-Confidence Curve'),
    ('confusion_matrix.png',           'Confusion Matrix'),
    ('confusion_matrix_normalized.png','Confusion Matrix (Normalized)'),
]

for fname, title in plot_files:
    fpath = RUN_DIR / fname
    if fpath.exists():
        print(f'\n{"=" * 60}')
        print(f'  {title}')
        print(f'{"=" * 60}')
        display(IPImage(filename=str(fpath), width=900))
    else:
        print(f'  {fname} not found')

In [ ]:
# ============================================================
# 2b. SAMPLE PREDICTIONS (val batch)
# ============================================================
pred_files = [
    ('val_batch0_labels.jpg', 'val_batch0_pred.jpg'),
    ('val_batch1_labels.jpg', 'val_batch1_pred.jpg'),
    ('val_batch2_labels.jpg', 'val_batch2_pred.jpg'),
]

for labels_f, preds_f in pred_files:
    labels_path = RUN_DIR / labels_f
    preds_path  = RUN_DIR / preds_f
    if labels_path.exists() and preds_path.exists():
        print(f'\n--- {labels_f.replace(".jpg", "")} ---')
        print('Ground Truth:')
        display(IPImage(filename=str(labels_path), width=800))
        print('Predictions:')
        display(IPImage(filename=str(preds_path), width=800))

---
## 3. Per-Class Checkpoint Eval (AP50 at key epochs)

In [ ]:
# ============================================================
# 3. PER-CLASS CHECKPOINT EVAL
# Runs model.val() on key checkpoints to extract per-class AP50.
# ============================================================

def eval_checkpoint(weights_path, label):
    """Run val on a checkpoint and return per-class metrics."""
    if not weights_path.exists():
        print(f'  [{label}] {weights_path.name} not found \u2014 skipping')
        return None

    model = YOLO(str(weights_path))
    m = model.val(
        data    = str(DATA_YAML),
        imgsz   = cfg['imgsz'],
        half    = cfg['half'],
        device  = 0,
        split   = 'val',
        verbose = False,
    )

    row = {
        'checkpoint': label,
        'map50':      float(m.box.map50),
        'map5095':    float(m.box.map),
    }
    for i, cls in enumerate(CLASS_NAMES):
        row[f'ap50_{cls}'] = float(m.box.ap50[i]) if i < len(m.box.ap50) else 0.0
        row[f'r_{cls}']    = float(m.box.r[i])    if i < len(m.box.r)    else 0.0
        row[f'p_{cls}']    = float(m.box.p[i])    if i < len(m.box.p)    else 0.0
    return row


# ---- Evaluate key checkpoints ----
checkpoints = [
    (WEIGHTS_DIR / 'epoch5.pt',  'ep5'),
    (WEIGHTS_DIR / 'epoch10.pt', 'ep10'),
    (WEIGHTS_DIR / 'epoch20.pt', 'ep20'),
    (WEIGHTS_DIR / 'epoch50.pt', 'ep50'),
    (WEIGHTS_DIR / 'best.pt',    'best'),
]

rows = []
for wpath, label in checkpoints:
    print(f'\nEvaluating {label} ({wpath.name})...')
    result = eval_checkpoint(wpath, label)
    if result is not None:
        rows.append(result)

# ---- Build results table ----
ckpt_df = pd.DataFrame(rows)

# Save to CSV
ckpt_log = RESULTS_DIR / 'nb04_checkpoint_evals.csv'
ckpt_df.to_csv(ckpt_log, index=False)
print(f'\nSaved -> {ckpt_log}')

# Display table
display_cols = ['checkpoint', 'map50', 'map5095'] + [f'ap50_{c}' for c in CLASS_NAMES]
display_cols = [c for c in display_cols if c in ckpt_df.columns]
show = ckpt_df[display_cols].copy()
show.columns = ['ckpt', 'mAP50', 'mAP50-95'] + CLASS_NAMES
print('\nCheckpoint Progression:')
print(show.to_string(index=False, float_format='{:.3f}'.format))

# Minority trend
print('\nMinority avg AP50 trend (vest + no-vest):')
for _, r in ckpt_df.iterrows():
    avg = (r.get('ap50_vest', 0) + r.get('ap50_no-vest', 0)) / 2
    bar = '\u2588' * int(avg * 40)
    print(f"  {r['checkpoint']:<8}: {avg:.3f}  {bar}")

---
## 4. Ablation Decision Gate

In [ ]:
# ============================================================
# 4. ABLATION DECISION GATE
# Based on best.pt minority class AP50.
# ============================================================
best_row = ckpt_df[ckpt_df['checkpoint'] == 'best'].iloc[0]
vest_ap50    = best_row.get('ap50_vest', 0.0)
novest_ap50  = best_row.get('ap50_no-vest', 0.0)
minority_avg = (vest_ap50 + novest_ap50) / 2

print(f'best.pt minority class results:')
print(f'  vest    AP50 : {vest_ap50:.3f}')
print(f'  no-vest AP50 : {novest_ap50:.3f}')
print(f'  average      : {minority_avg:.3f}')
print()

if minority_avg >= 0.50:
    print('DECISION: OK (>= 0.50)')
    print('  No ablation needed. Proceed to NB05 (YOLO11s).')
elif 0.30 <= minority_avg < 0.50:
    print('DECISION: MARGINAL (0.30 - 0.50)')
    print('  Recommended: run Scenario A (copy_paste 0.35) in NB04 Section 6')
else:
    print('DECISION: INTERVENE (< 0.30)')
    print('  Run Scenario B (fl_gamma 1.5) in NB04 Section 6')

---
## 5. Final Evaluation (val + test) → NB07 comparison table

In [ ]:
# ============================================================
# 5. FINAL EVALUATION — val + test splits
# Evaluates best.pt on both splits and saves to
# results/nb07_model_comparison.csv for the NB07 comparison.
# ============================================================
best_weights = WEIGHTS_DIR / 'best.pt'
assert best_weights.exists(), f'best.pt not found in {WEIGHTS_DIR}'

final_model = YOLO(str(best_weights))
final_results = {}

for split in ('val', 'test'):
    m = final_model.val(
        data    = str(DATA_YAML),
        imgsz   = cfg['imgsz'],
        half    = cfg['half'],
        device  = 0,
        split   = split,
        verbose = False,
    )
    final_results[split] = m

    print(f'\n{"=" * 62}')
    print(f'  {split.upper()} SET \u2014 best.pt ({RUN_NAME})')
    print(f'{"=" * 62}')
    print(f'  {"Class":<14} {"AP50":>8} {"AP50-95":>10} {"Precision":>10} {"Recall":>8}')
    print(f'  {"-"*56}')
    for i, cls_name in enumerate(CLASS_NAMES):
        ap50 = float(m.box.ap50[i]) if i < len(m.box.ap50) else 0.0
        ap   = float(m.box.ap[i])   if i < len(m.box.ap)   else 0.0
        p    = float(m.box.p[i])    if i < len(m.box.p)    else 0.0
        r    = float(m.box.r[i])    if i < len(m.box.r)    else 0.0
        flag = '  <- MINORITY' if cls_name in MINORITY_CLS else ''
        print(f'  {cls_name:<14} {ap50:>8.3f} {ap:>10.3f} {p:>10.3f} {r:>8.3f}{flag}')
    print(f'  {"-"*56}')
    print(f'  {"ALL":<14} {float(m.box.map50):>8.3f} {float(m.box.map):>10.3f}')

# ---- Persist for NB07 comparison table ----
save_record = {'model': RUN_NAME, 'imgsz': cfg['imgsz']}
for split in ('val', 'test'):
    m = final_results[split]
    save_record[f'{split}_map50']   = float(m.box.map50)
    save_record[f'{split}_map5095'] = float(m.box.map)
    for i, cls_name in enumerate(CLASS_NAMES):
        save_record[f'{split}_ap50_{cls_name}'] = (
            float(m.box.ap50[i]) if i < len(m.box.ap50) else 0.0
        )

nb07_log = RESULTS_DIR / 'nb07_model_comparison.csv'
new_row  = pd.DataFrame([save_record])
if nb07_log.exists():
    existing = pd.read_csv(nb07_log)
    existing = existing[existing['model'] != RUN_NAME]
    pd.concat([existing, new_row], ignore_index=True).to_csv(nb07_log, index=False)
else:
    new_row.to_csv(nb07_log, index=False)

print(f'\nFinal results saved -> {nb07_log}')
print(f'NB05 and NB06 will append to this same file.')

---
## Summary & Handoff to NB05

**Before moving to NB05, confirm:**

- [ ] Training curves look healthy (losses decreasing, metrics plateauing)
- [ ] Ablation decision made (Section 4 above)
- [ ] `nb07_model_comparison.csv` has the YOLO11n row
- [ ] If ablation was needed: run it in NB04 Section 6, then re-run Section 5 above

```python
# NB05 opening:
cfg = ALL_CONFIGS['yolo11s']   # imgsz=1280, batch=4, nbs=64
model = YOLO('yolo11s.pt')
```